# drift on a GPU: does real FP8 agree with the simulator?

Everything in [drift](https://github.com/aarohigandhi/drift) is simulated arithmetic in
Java, checked against exact rational arithmetic. That proves the simulator is
self consistent. It does not prove it matches what a GPU actually does.

This notebook checks three claims on real `torch.float8_e4m3fn` casts:

1. The OCP shared exponent rule clamps **27 to 30%** of tanh activations in the MLP.
2. It clamps only about **2%** of attention probabilities, a tenth as much.
3. With that rule, FP8 attention training **diverges**, and one extra bit of
   exponent headroom rescues it.

Runtime: pick a GPU runtime. T4 is enough. Total run time is a few minutes.


In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert hasattr(torch, "float8_e4m3fn"), "this torch build has no float8_e4m3fn"
E4M3_MAX = 448.0          # largest finite value of E4M3
E4M3_EMAX = 8             # exponent of that value
print("device:", DEV)

## The corpus

The same text the repository trains on: the Python language reference, which ships
inside every Python install, so nothing is downloaded.

In [ ]:
import pydoc_data.topics as topics

text = "\n\n".join(topics.topics[k] for k in sorted(topics.topics))
text = text.encode("ascii", "ignore").decode("ascii")
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(len(data) * 0.9)
train_data, val_data = data[:split], data[split:]
VOCAB = len(chars)
print(f"{len(text):,} chars, vocab {VOCAB}")

## The OCP shared exponent cast, in PyTorch

One shared power of two per block of 32 values, chosen as
`floor(log2(blockmax)) - emax + headroom`. With headroom 0 this is the
specification, and it maps a block maximum into a range that E4M3 cannot fully
reach, so the largest values in a block clamp. The element cast is a real
`float8_e4m3fn` conversion, not a simulation.

In [ ]:
def mx_cast(x, block=32, headroom=0):
    """Cast x through an MX style shared exponent. Returns (values, clamped fraction)."""
    shape, dtype = x.shape, x.dtype
    flat = x.reshape(-1).float()
    n = flat.numel()
    pad = (-n) % block
    if pad:
        flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, block)

    amax = blocks.abs().amax(dim=1, keepdim=True)
    nonzero = amax > 0
    shared = torch.zeros_like(amax)
    shared[nonzero] = torch.floor(torch.log2(amax[nonzero])) - E4M3_EMAX + headroom
    shared = shared.clamp(-127, 127)
    scale = torch.pow(torch.tensor(2.0, device=x.device), shared)

    scaled = blocks / scale
    clamped = (scaled.abs() > E4M3_MAX) & nonzero
    q = scaled.clamp(-E4M3_MAX, E4M3_MAX).to(torch.float8_e4m3fn).float()
    out = (q * scale).view(-1)[:n].view(shape).to(dtype)
    return out, clamped.float().mean().item()


# Sanity check against the repository's arithmetic: a block whose maximum is just
# under a power of two pushes everything above 448/512 of it over the limit.
probe = torch.full((32,), 0.99, device=DEV)
probe[0] = 0.999
_, frac = mx_cast(probe, headroom=0)
print(f"contrived block, spec exponent: {frac * 100:.0f}% clamped")
_, frac_h = mx_cast(probe, headroom=1)
print(f"contrived block, one bit of headroom: {frac_h * 100:.0f}% clamped")

## Claim 1 and 2: clamp rates on real activations

An MLP with tanh, and an attention block, both trained briefly in fp32. Then the
clamp rate is measured on the activations each one actually produces.

In [ ]:
def batches(split_data, ctx, batch, steps, seed=0):
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        ix = torch.randint(len(split_data) - ctx - 1, (batch,), generator=g)
        x = torch.stack([split_data[i:i + ctx] for i in ix]).to(DEV)
        y = torch.stack([split_data[i + 1:i + ctx + 1] for i in ix]).to(DEV)
        yield x, y


class Mlp(nn.Module):
    """The repository's MLP: embed a window, two tanh layers, softmax."""
    def __init__(self, ctx=8, emb=16, hidden=128):
        super().__init__()
        self.ctx = ctx
        self.emb = nn.Embedding(VOCAB, emb)
        self.l1 = nn.Linear(ctx * emb, hidden, bias=True)
        self.l2 = nn.Linear(hidden, hidden, bias=True)
        self.l3 = nn.Linear(hidden, VOCAB, bias=True)
        self.acts = []

    def forward(self, idx, keep=False):
        h = self.emb(idx).flatten(1)
        a1 = torch.tanh(self.l1(h))
        a2 = torch.tanh(self.l2(a1))
        if keep:
            self.acts = [a1.detach(), a2.detach()]
        return self.l3(a2)


class Attn(nn.Module):
    """One causal attention block plus an MLP, no layer norm, as in the repository."""
    def __init__(self, ctx=16, dim=32, hidden=64):
        super().__init__()
        self.ctx, self.dim = ctx, dim
        self.emb = nn.Embedding(VOCAB, dim)
        self.pos = nn.Parameter(torch.zeros(ctx, dim))
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.o = nn.Linear(dim, dim, bias=False)
        self.f1 = nn.Linear(dim, hidden)
        self.f2 = nn.Linear(hidden, dim)
        self.head = nn.Linear(dim, VOCAB)
        self.probs = None

    def forward(self, idx, keep=False):
        b, t = idx.shape
        x = self.emb(idx) + self.pos[:t]
        q, k, v = self.q(x), self.k(x), self.v(x)
        att = (q @ k.transpose(1, 2)) / math.sqrt(self.dim)
        mask = torch.tril(torch.ones(t, t, device=idx.device, dtype=torch.bool))
        att = att.masked_fill(~mask, float("-inf"))
        p = att.softmax(-1)
        if keep:
            self.probs = p.detach()[mask.expand(b, t, t)]   # the real probabilities only
        h = x + self.o(p @ v)
        y = h + self.f2(F.relu(self.f1(h)))
        return self.head(y)


def train_fp32(model, ctx, steps=600, batch=32, lr=2e-3, seed=0):
    model.to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for x, y in batches(train_data, ctx, batch, steps, seed):
        logits = model(x)
        target = y[:, -1] if isinstance(model, Mlp) else y
        loss = F.cross_entropy(logits.reshape(-1, VOCAB) if logits.dim() == 3 else logits,
                               target.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()


torch.manual_seed(1)
mlp = Mlp()
print("mlp final loss", round(train_fp32(mlp, 8), 3))
torch.manual_seed(1)
attn = Attn()
print("attention final loss", round(train_fp32(attn, 16), 3))

In [ ]:
# Clamp rates on what each model actually produces.
x, _ = next(iter(batches(val_data, 8, 256, 1, seed=7)))
mlp(x, keep=True)
tanh_rates = [mx_cast(a, headroom=0)[1] for a in mlp.acts]

x2, _ = next(iter(batches(val_data, 16, 256, 1, seed=7)))
attn(x2, keep=True)
prob_rate = mx_cast(attn.probs, headroom=0)[1]

tanh_rates_h = [mx_cast(a, headroom=1)[1] for a in mlp.acts]
prob_rate_h = mx_cast(attn.probs, headroom=1)[1]

print(f"tanh activations clamped, spec exponent : {100 * sum(tanh_rates) / len(tanh_rates):.1f}%"
      f"   (drift simulator: 27 to 30%)")
print(f"attention probabilities clamped, spec    : {100 * prob_rate:.1f}%"
      f"   (drift simulator: 1.9%)")
print(f"tanh activations, one bit of headroom    : {100 * sum(tanh_rates_h) / len(tanh_rates_h):.2f}%")
print(f"attention probabilities, headroom        : {100 * prob_rate_h:.2f}%")

## Claim 3: does FP8 attention training diverge on a GPU?

Every linear layer, and both attention matmuls, cast their inputs through the same
MX rule. The forward pass uses the cast values and the backward pass is left to
autograd through a straight through estimator, which is the usual way to simulate a
low precision forward on a GPU. The repository casts the backward operands too, so
this is a weaker test: if it still diverges, the effect is not an artifact of the
Java implementation.

In [ ]:
class CastFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, headroom):
        out, _ = mx_cast(x, headroom=headroom)
        return out

    @staticmethod
    def backward(ctx, g):
        return g, None


def cast(x, headroom):
    return CastFn.apply(x, headroom)


class AttnFp8(Attn):
    def __init__(self, headroom, **kw):
        super().__init__(**kw)
        self.headroom = headroom

    def forward(self, idx, keep=False):
        b, t = idx.shape
        hr = self.headroom
        x = self.emb(idx) + self.pos[:t]
        xc = cast(x, hr)
        q = F.linear(xc, cast(self.q.weight, hr))
        k = F.linear(xc, cast(self.k.weight, hr))
        v = F.linear(xc, cast(self.v.weight, hr))
        att = (cast(q, hr) @ cast(k, hr).transpose(1, 2)) / math.sqrt(self.dim)
        mask = torch.tril(torch.ones(t, t, device=idx.device, dtype=torch.bool))
        att = att.masked_fill(~mask, float("-inf"))
        p = att.softmax(-1)
        ctxv = cast(p, hr) @ cast(v, hr)
        h = x + F.linear(cast(ctxv, hr), cast(self.o.weight, hr))
        m = F.relu(F.linear(cast(h, hr), cast(self.f1.weight, hr), self.f1.bias))
        y = h + F.linear(cast(m, hr), cast(self.f2.weight, hr), self.f2.bias)
        return F.linear(cast(y, hr), cast(self.head.weight, hr), self.head.bias)


def run(headroom, steps=1200, seed=1, lr=2e-3, batch=8):
    torch.manual_seed(seed)
    model = AttnFp8(headroom=headroom).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    blew_up_at = None
    for i, (x, y) in enumerate(batches(train_data, 16, batch, steps, seed), start=1):
        loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1))
        if not torch.isfinite(loss) or loss.item() > 2 * math.log(VOCAB):
            blew_up_at = i
            break
        opt.zero_grad(); loss.backward(); opt.step()
    return blew_up_at, loss.item()


for seed in (1, 2, 3):
    spec = run(headroom=0, seed=seed)
    head = run(headroom=1, seed=seed)
    print(f"seed {seed}:  spec exponent -> {'diverged at step ' + str(spec[0]) if spec[0] else f'ok, loss {spec[1]:.3f}'}"
          f"   |  one bit headroom -> {'diverged at step ' + str(head[0]) if head[0] else f'ok, loss {head[1]:.3f}'}")

## What to report

Paste the three numbers back into the repository:

1. tanh clamp rate against the simulator's 27 to 30%
2. attention probability clamp rate against the simulator's 1.9%
3. whether the spec exponent diverges on a GPU and headroom survives, against the
   simulator's divergence at steps 446, 453 and 497

If a number disagrees, that is the interesting outcome and it goes in the README
exactly as measured. The simulator casts both forward and backward operands and
this notebook casts only the forward ones, so the divergence here is expected to
be no earlier than in the simulator, and possibly absent. Say which happened.